In [2]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


In [3]:
df2 = pd.read_csv(r"C:\Users\HP\Desktop\csv4.csv")

row_ids = df2['Row ID'].copy()

df2 = df2.drop(columns=['Row ID', 'Country'])

categorical_cols = ['Order Priority', 'Category', 'Sub-Category', 'Segment', 'Market', 'Region', 'Day Name', 'Month Name', 'Is Weekend']
df2 = pd.get_dummies(df2, columns=categorical_cols, dtype=int)

le = LabelEncoder()
df2['Ship Mode'] = le.fit_transform(df2['Ship Mode'])

X = df2.drop(columns=['Ship Mode'])
y = df2['Ship Mode']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=0, stratify=y)

cust_order_cnt = X_train.groupby('Customer Key')['Customer Key'].count()
cust_avg_sales = X_train.groupby('Customer Key')['Sales'].mean()
global_avg_sales = X_train['Sales'].mean()

X_train['Cust_Order_Count'] = X_train['Customer Key'].map(cust_order_cnt).fillna(1)
X_train['Cust_Avg_Sales'] = X_train['Customer Key'].map(cust_avg_sales).fillna(global_avg_sales)

X_test['Cust_Order_Count'] = X_test['Customer Key'].map(cust_order_cnt).fillna(1)
X_test['Cust_Avg_Sales'] = X_test['Customer Key'].map(cust_avg_sales).fillna(global_avg_sales)

X_train = X_train.drop(columns=['Customer Key'])
X_test = X_test.drop(columns=['Customer Key'])


In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=0)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)


C:\Users\HP\AppData\Roaming\Python\Python39\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [6]:
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(X_train_res, y_train_res.to_numpy().ravel())

y_pred_knn = knn.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred_knn)
prec = precision_score(y_test, y_pred_knn, average='weighted')
rec = recall_score(y_test, y_pred_knn, average='weighted')
f1 = f1_score(y_test, y_pred_knn, average='weighted')

print(f"accuracy:  {round(100*acc, 1)}%")
print(f"precision: {round(100*prec, 1)}%")
print(f"recall:    {round(100*rec, 1)}%")
print(f"f1:        {round(100*f1, 1)}%")
print(classification_report(y_test, y_pred_knn))

cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_knn_df = pd.DataFrame(cm_knn, index=le.classes_, columns=le.classes_)
cm_knn_df


accuracy:  46.3%
precision: 49.7%
recall:    46.3%
f1:        47.2%
              precision    recall  f1-score   support

           0       0.40      0.40      0.40      1446
           1       0.20      0.43      0.27       524
           2       0.51      0.37      0.43      1995
           3       0.63      0.61      0.62      2035

    accuracy                           0.46      6000
   macro avg       0.43      0.45      0.43      6000
weighted avg       0.50      0.46      0.47      6000



,First Class,Same Day,Second Class,Standard Class
First Class,585,315,311,235
Same Day,124,225,104,71
Second Class,473,365,732,425
Standard Class,284,231,283,1237


In [7]:
svc = SVC(C=30.0, kernel='rbf', tol=1e-2, random_state=0)
svc.fit(X_train_res, y_train_res.to_numpy().ravel())

y_pred_svc = svc.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred_svc)
prec = precision_score(y_test, y_pred_svc, average='weighted')
rec = recall_score(y_test, y_pred_svc, average='weighted')
f1 = f1_score(y_test, y_pred_svc, average='weighted')

print(f"accuracy:  {round(100*acc, 1)}%")
print(f"precision: {round(100*prec, 1)}%")
print(f"recall:    {round(100*rec, 1)}%")
print(f"f1:        {round(100*f1, 1)}%")
print(classification_report(y_test, y_pred_svc))

cm_svc = confusion_matrix(y_test, y_pred_svc)
cm_svc_df = pd.DataFrame(cm_svc, index=le.classes_, columns=le.classes_)
cm_svc_df


accuracy:  55.7%
precision: 55.5%
recall:    55.7%
f1:        55.5%
              precision    recall  f1-score   support

           0       0.43      0.47      0.45      1446
           1       0.36      0.31      0.33       524
           2       0.55      0.51      0.53      1995
           3       0.70      0.73      0.71      2035

    accuracy                           0.56      6000
   macro avg       0.51      0.50      0.51      6000
weighted avg       0.55      0.56      0.56      6000



,First Class,Same Day,Second Class,Standard Class
First Class,678,111,438,219
Same Day,154,162,138,70
Second Class,499,118,1021,357
Standard Class,236,60,258,1481


In [8]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=0)
rf.fit(X_train_res, y_train_res.to_numpy().ravel())

y_pred_rf = rf.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred_rf)
prec = precision_score(y_test, y_pred_rf, average='weighted')
rec = recall_score(y_test, y_pred_rf, average='weighted' )
f1 = f1_score(y_test, y_pred_rf, average='weighted')

print(f"accuracy:  {round(100*acc, 1)}%")
print(f"precision: {round(100*prec, 1)}%")
print(f"recall:    {round(100*rec, 1)}%")
print(f"f1:        {round(100*f1, 1)}%")
print(classification_report(y_test, y_pred_rf))

cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_rf_df = pd.DataFrame(cm_rf, index=le.classes_, columns=le.classes_)
cm_rf_df


accuracy:  57.9%
precision: 55.7%
recall:    57.9%
f1:        54.7%
              precision    recall  f1-score   support

           0       0.45      0.38      0.41      1446
           1       0.26      0.18      0.21       524
           2       0.62      0.40      0.49      1995
           3       0.65      0.99      0.79      2035

    accuracy                           0.58      6000
   macro avg       0.49      0.49      0.47      6000
weighted avg       0.56      0.58      0.55      6000



,First Class,Same Day,Second Class,Standard Class
First Class,552,144,383,367
Same Day,184,94,111,135
Second Class,491,120,804,580
Standard Class,5,5,2,2023


In [9]:
ada = AdaBoostClassifier(n_estimators=100, learning_rate=0.1, algorithm='SAMME', random_state=0)
ada.fit(X_train_res, y_train_res.to_numpy().ravel())

y_pred_ada = ada.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred_ada)
prec = precision_score(y_test, y_pred_ada, average='weighted')
rec = recall_score(y_test, y_pred_ada, average='weighted')
f1 = f1_score(y_test, y_pred_ada, average='weighted')

print(f"accuracy:  {round(100*acc, 1)}%")
print(f"precision: {round(100*prec, 1)}%")
print(f"recall:    {round(100*rec, 1)}%")
print(f"f1:        {round(100*f1, 1)}%")
print(classification_report(y_test, y_pred_ada))

cm_ada = confusion_matrix(y_test, y_pred_ada)
cm_ada_df = pd.DataFrame(cm_ada, index=le.classes_, columns=le.classes_)
cm_ada_df


C:\Users\HP\AppData\Roaming\Python\Python39\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


accuracy:  48.5%
precision: 42.3%
recall:    48.5%
f1:        41.3%
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1446
           1       0.16      0.48      0.24       524
           2       0.62      0.31      0.42      1995
           3       0.60      1.00      0.75      2035

    accuracy                           0.48      6000
   macro avg       0.34      0.45      0.35      6000
weighted avg       0.42      0.48      0.41      6000



C:\Users\HP\AppData\Roaming\Python\Python39\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\HP\AppData\Roaming\Python\Python39\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\HP\AppData\Roaming\Python\Python39\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(resu

,First Class,Same Day,Second Class,Standard Class
First Class,0,674,297,475
Same Day,0,250,84,190
Second Class,0,668,626,701
Standard Class,0,0,1,2034


In [10]:
train_acc = best_xgb.score(X_train_res, y_train_res)
test_acc = best_xgb.score(X_test_scaled, y_test)

print("Train Accuracy:", round(100 * train_acc, 1), "%")
print("Test Accuracy:", round(100 * test_acc, 1), "%")


Train Accuracy: 75.3 %
Test Accuracy: 62.4 %


In [11]:
best_xgb = xgb.XGBClassifier(
    booster="gbtree",
    n_estimators=150,
    learning_rate=0.1,
    max_depth=10,
    reg_alpha=1,
    reg_lambda=1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=0
)
best_xgb.fit(X_train_res, y_train_res.to_numpy().ravel())

y_pred_xgb = best_xgb.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred_xgb)
prec = precision_score(y_test, y_pred_xgb, average='weighted' )
rec = recall_score(y_test, y_pred_xgb, average='weighted')
f1 = f1_score(y_test, y_pred_xgb, average='weighted')

print(f"accuracy:  {round(100*acc, 1)}%")
print(f"precision: {round(100*prec, 1)}%")
print(f"recall:    {round(100*rec, 1)}%")
print(f"f1:        {round(100*f1, 1)}%")
print(classification_report(y_test, y_pred_xgb))

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
cm_xgb_df = pd.DataFrame(cm_xgb, index=le.classes_, columns=le.classes_)
cm_xgb_df

accuracy:  66.5%
precision: 66.3%
recall:    66.5%
f1:        64.4%
              precision    recall  f1-score   support

           0       0.62      0.47      0.54      1446
           1       0.73      0.23      0.35       524
           2       0.65      0.63      0.64      1995
           3       0.69      0.95      0.80      2035

    accuracy                           0.67      6000
   macro avg       0.67      0.57      0.58      6000
weighted avg       0.66      0.67      0.64      6000



,First Class,Same Day,Second Class,Standard Class
First Class,685,24,432,305
Same Day,109,122,183,110
Second Class,265,18,1255,457
Standard Class,38,4,63,1930


In [12]:
train_acc = best_xgb.score(X_train_res, y_train_res)
test_acc = best_xgb.score(X_test_scaled, y_test)

print("Train Accuracy:", round(100 * train_acc, 1), "%")
print("Test Accuracy:", round(100 * test_acc, 1), "%")


Train Accuracy: 94.1 %
Test Accuracy: 66.5 %


In [ ]:
feature_names = X_train.columns

df_importance = pd.DataFrame({
    'Feature': feature_names,
    'RandomForest': rf.feature_importances_,
    'AdaBoost': ada.feature_importances_,
    'XGBoost': best_xgb.feature_importances_
})

df_importance = df_importance.sort_values(by='XGBoost', ascending=False).reset_index(drop=True)
df_importance.head(15)
